# 📌 MODUL 02 — Pertemuan 03: Arsitektur Apache Hadoop & Sistem Berkas HDFS
> **Tujuan Pembelajaran:** Memahami kerangka 5V Big Data, arsitektur Master-Worker HDFS (NameNode, DataNode, Secondary NameNode), YARN, dan menguji langsung seluruh perintah CLI HDFS.

## 🌟 1. Fondasi Teori: 5V Big Data & Arsitektur HDFS

Dalam ujian lisan atau kuis responsi, dosen sering menanyakan konsep ini:
1. **5V Karakteristik Big Data:**
   - **Volume:** Ukuran data super masif (terabyte hingga petabyte).
   - **Velocity:** Kecepatan masuknya data secara real-time (ribuan transaksi/detik).
   - **Variety:** Keberagaman format (terstruktur seperti CSV/tabel, semi-terstruktur seperti JSON, tak terstruktur seperti video/teks).
   - **Veracity:** Kualitas dan kebersihan data mentah (sering ada data corrupt/null).
   - **Value:** Nilai wawasan ekonomi yang dapat diekstrak untuk bisnis.

2. **Arsitektur HDFS (Master-Worker):**
   - **NameNode (Master):** Mengelola metadata direktori, nama berkas, dan memetakan lokasi blok. **TIDAK menyimpan isi berkas fisik**.
   - **DataNode (Worker):** Menyimpan potongan fisik data aktual (default blok 128 MB) dan mengirim detak jantung (*heartbeat*) tiap 3 detik ke NameNode.
   - **Secondary NameNode:** Melakukan *checkpointing* berkala menggabungkan `fsimage` dan `edit logs` agar boot NameNode tidak lambat. (Bukan failover/cadangan pengganti langsung!).

## 🚦 2. Memeriksa Status Klaster Hadoop di Laptop
Sebelum menjalankan perintah HDFS, pastikan klaster Hadoop sudah berjalan. Jika belum, buka terminal dan ketik:
```bash
sudo systemctl start sshd
start-dfs.sh
start-yarn.sh
```
Jalankan sel berikut untuk memeriksa proses Java yang sedang aktif di laptopmu:

In [ ]:
# Memeriksa daemon Hadoop yang aktif
!jps

*Output sehat harus berisi: `NameNode`, `DataNode`, `SecondaryNameNode`, `ResourceManager`, `NodeManager`.*

Dashboard Monitoring Web:
- **NameNode Web UI:** [http://localhost:9870](http://localhost:9870)
- **YARN Resource Manager:** [http://localhost:8088](http://localhost:8088)

## 🛠️ 3. Menyiapkan Dataset Lokal untuk Pengujian HDFS
Kita buat berkas CSV contoh sederhana di lingkungan lokal untuk kita unggah ke HDFS:

In [ ]:
import pandas as pd
import numpy as np

# Buat berkas CSV lokal
np.random.seed(42)
df_dummy = pd.DataFrame({
    "transaksi_id": [f"TX-{i:04d}" for i in range(1, 21)],
    "cabang": np.random.choice(["Magelang", "Yogyakarta", "Semarang", "Solo"], size=20),
    "total_belanja": np.random.randint(50000, 1500000, size=20)
})

df_dummy.to_csv("transaksi_latihan_p3.csv", index=False)
print(f"✅ Berkas lokal 'transaksi_latihan_p3.csv' berhasil dibuat dengan {len(df_dummy)} baris!")
df_dummy.head()

## 💻 4. Hands-On: Perintah Dasar HDFS CLI

Format umum: `hdfs dfs -[perintah] [argumen]`

### 4.1 Membuat Folder di HDFS (`-mkdir`)
Opsi `-p` membuat folder induk secara otomatis jika belum ada:

In [ ]:
!hdfs dfs -mkdir -p /user/$USER/responsi/p3/raw
print("✅ Folder /user/$USER/responsi/p3/raw berhasil dibuat di HDFS!")

### 4.2 Mengunggah Berkas dari Lokal ke HDFS (`-put`)
Opsi `-f` memaksa timpa (*force overwrite*) jika berkas sudah ada:

In [ ]:
!hdfs dfs -put -f transaksi_latihan_p3.csv /user/$USER/responsi/p3/raw/
print("✅ Berkas berhasil diunggah ke HDFS!")

### 4.3 Melihat Daftar Isi Folder HDFS (`-ls`)
Opsi `-h` menampilkan ukuran berkas dalam format ramah manusia (KB, MB):

In [ ]:
!hdfs dfs -ls -h /user/$USER/responsi/p3/raw/

### 4.4 Mengintip Isi Berkas Langsung dari HDFS (`-cat`)
Selalu kombinasikan dengan `head` agar terminal tidak banjir jutaan baris:

In [ ]:
!hdfs dfs -cat /user/$USER/responsi/p3/raw/transaksi_latihan_p3.csv | head -n 6

### 4.5 Memeriksa Kapasitas Ruang Disk HDFS (`-du`)
Mengetahui seberapa besar ruang HDFS yang dikonsumsi berkas/folder:

In [ ]:
!hdfs dfs -du -h /user/$USER/responsi/p3/

### 4.6 Mengunduh Berkas dari HDFS ke Disk Lokal Laptop (`-get`)

In [ ]:
!hdfs dfs -get -f /user/$USER/responsi/p3/raw/transaksi_latihan_p3.csv ./transaksi_p3_unduhan.csv
!ls -lh transaksi_p3_unduhan.csv
print("✅ Berkas sukses diunduh kembali dari HDFS ke lokal laptop!")

### 4.7 Menghapus Berkas atau Folder di HDFS (`-rm`)
Opsi `-r` untuk menghapus folder rekursif:

In [ ]:
# Uji coba hapus berkas unduhan lokal
!rm -f transaksi_latihan_p3.csv transaksi_p3_unduhan.csv
print("✅ Berkas lokal sampah dibersihkan.")

# Hapus data uji coba di HDFS (uncomment jika ingin mencoba)
# !hdfs dfs -rm -r /user/$USER/responsi/p3/raw/transaksi_latihan_p3.csv

---
## 🎓 Tanya-Jawab Khas Responsi (Bocoran Pertanyaan Dosen)
1. **Kenapa replikasi di laptop bernilai 1 di `hdfs-site.xml`?**  
   *Karena kita menggunakan mode single-node (pseudo-distributed). Di industri server nyata nilainya 3 agar jika ada 2 server terbakar, data tetap aman di server ketiga.*
2. **Jika file 200 MB diunggah ke HDFS dengan blok 128 MB, terbagi berapa blok?**  
   *Terbagi menjadi 2 blok: Blok 1 berukuran 128 MB, dan Blok 2 berukuran 72 MB (hanya memakan 72 MB fisik, tidak membuang sisa 128 MB).*